# `tracker` — clustering `detectorsim2d` hits

`tracker` turns `detectorsim2d` tracker hits into clusters via 1D
adjacent-cell grouping along each layer — the tracking-detector analogue of
`sensor`'s 2D pixel-grid clustering. It's deliberately 1D and
dependency-light (no scipy): a single layer's hit cells are just a sorted
sequence, so "adjacent" is a gap test between consecutive sorted cell
indices.

Pipeline: `detectorsim2d.simulate_events` -> `tracker.digitize_hits` (adds
`cell_index`) -> `tracker.cluster_hits` (adds `cluster_id`, aggregates).


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from detectorsim2d.config import load_config
from detectorsim2d.simulate import simulate_events
from tracker.digitize import digitize_hits
from tracker.clustering import cluster_hits


`barrel6.yaml` is a 6-layer barrel tracker with per-layer `pitch` set, which
is what makes digitization possible.

Note: `simulate_events` returns a **3-tuple** `(particles, hits, deposits)`
— the `tracker` README's own snippet unpacks only `(particles, hits)`,
which is stale against the current `detectorsim2d` API.


In [ ]:
config = load_config("../aihep/simulator/detectorsim2d/configs/barrel6.yaml")
particles, hits, deposits = simulate_events(config, rng=np.random.default_rng(7))
print("hits:", hits.shape)
hits.head()


## Digitizing: `cell_index = floor(s_local / pitch)`

Layers with no `pitch` set are left un-digitized (`cell_index = NaN`) and
will end up excluded from clustering.


In [ ]:
digitized = digitize_hits(hits, config.layers)
digitized[["layer_id", "s_local", "cell_index"]].head()


## Clustering: 1D connected components per `(event_id, layer_id)`

`cluster_hits(hits_df, connectivity_gap=1)` merges hits whose digitized
cell indices are within `connectivity_gap` of each other, then aggregates
per-cluster centroids. `cluster_id` is numbered uniquely **within an event,
across all of its layers** — not restarted per layer.


In [ ]:
clustered_hits, clusters = cluster_hits(digitized)
print("clustered hits:", clustered_hits.shape, "clusters:", clusters.shape)
clusters.head()


In [ ]:
event0 = clustered_hits[clustered_hits["event_id"] == 0]

fig, ax = plt.subplots(figsize=(6, 6))
for cluster_id, group in event0.groupby("cluster_id"):
    label = f"cluster {cluster_id}" if cluster_id >= 0 else "unclustered (no pitch)"
    ax.scatter(group["x"], group["y"], s=15, label=label)
ax.set_aspect("equal")
ax.set_title("Clustered tracker hits, event 0")
ax.legend(fontsize=6, loc="upper left", bbox_to_anchor=(1.02, 1))


Each color above is one cluster spanning (typically) a handful of adjacent
digitized cells on one layer — since `barrel6.yaml`'s modules are narrow and
`pitch` is fine relative to a track's spread, most clusters here are single
hits, exactly as expected for well-separated, low-occupancy tracks.
